<a href="https://colab.research.google.com/github/sarthakkalra/part-2-cnn-computer-vision/blob/main/notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import os
import zipfile
import pathlib
import shutil
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
import tensorflow as tf
import requests # Import requests for the new download function

print("TensorFlow version:", tf.__version__)

# --- Part 1: Download and Extract Dataset ---
print("\n--- Part 1: Downloading and Extracting Dataset ---")

# The actual file ID from the problematic URL in the previous run (17xoSIAe-24-18iJiN3zKPqJl-RNDqIeD)
# Note: The problem statement provided a folder URL (17xoSIAe-24-18iJiN3zKPqJl-RNDqIeW).
# Assuming the intention was to download a specific zip file within that folder,
# and the previous `gdown` attempt used an inferred file ID.
# We will use this file ID directly in the custom download function.
file_id_to_download = "17xoSIAe-24-18iJiN3zKPqJl-RNDqIeD"
zip_file_name = "dataset.zip"
data_dir_name = "animal_faces"

# Custom function to download from Google Drive, handling confirmation
def download_file_from_google_drive(id, destination):
    URL = "https://docs.google.com/uc?export=download"
    session = requests.Session()
    response = session.get(URL, params = { 'id' : id }, stream = True)
    token = None
    for key, value in response.cookies.items():
        if key.startswith('download_warning'):
            token = value
            break

    if token:
        params = { 'id' : id, 'confirm' : token }
        response = session.get(URL, params = params, stream = True)

    CHUNK_SIZE = 32768
    with open(destination, "wb") as f:
        for chunk in response.iter_content(CHUNK_SIZE):
            if chunk: # filter out keep-alive new chunks
                f.write(chunk)
    print(f"Successfully downloaded {destination}")

# Attempt to download the dataset using the custom function
try:
    download_file_from_google_drive(file_id_to_download, zip_file_name)
except Exception as e:
    print(f"Error downloading file: {e}")
    print("Please ensure the Google Drive file ID is correct and publicly accessible.")
    print("You may need to manually download the dataset and place it in the Colab environment.")
    raise # Re-raise the exception to stop execution if download fails

# Extract the dataset
with zipfile.ZipFile(zip_file_name, 'r') as zip_ref:
    zip_ref.extractall(".")

# Rename the extracted folder to a more manageable name if needed
# The extracted folder is likely named 'animal_faces' based on the prompt's context
# If it's different, you might need to adjust this.
if os.path.exists(data_dir_name):
    print(f"Dataset '{data_dir_name}' already exists.")
else:
    # Assuming the zip contains a top-level directory or multiple subdirectories
    # You might need to inspect the zip content or adjust this path
    # For instance, if it extracts to 'drive-download-2023...', then rename it.
    # For now, let's assume it extracts directly to a folder like 'animal_faces' or similar.
    extracted_content = os.listdir('.')
    potential_data_folder = [d for d in extracted_content if os.path.isdir(d) and 'animal' in d.lower()]
    if potential_data_folder:
        shutil.move(potential_data_folder[0], data_dir_name)
        print(f"Moved '{potential_data_folder[0]}' to '{data_dir_name}'.")
    else:
        print("Could not find a clear data directory after extraction. Please check the extracted content.")

data_dir = pathlib.Path(data_dir_name)
if not data_dir.exists():
    raise FileNotFoundError(f"Expected data directory '{data_dir_name}' not found. Please check extraction process.")

# --- Task 1: Problem Identification ---
print("\n--- Task 1: Problem Identification ---")
print("Based on the typical structure of image datasets provided for such tasks (folders representing categories of images),")
print("and the goal to learn visual patterns to distinguish images, this dataset represents an:")
print("**Image Classification** problem.")
print("This is appropriate because each image in the dataset will likely belong to a specific category (e.g., a type of animal or object), and the goal is to train a model to predict that category for unseen images.")


# --- Task 2: Dataset Exploration ---
print("\n--- Task 2: Dataset Exploration ---")

# Number of classes
class_names = sorted([item.name for item in data_dir.iterdir() if item.is_dir()])
num_classes = len(class_names)
print(f"Number of classes: {num_classes}")
print(f"Class names: {class_names}")

# Number of images per class and overall image count
image_counts = {}
total_images = 0
for class_name in class_names:
    class_path = data_dir / class_name
    images = list(class_path.glob('*.jpg')) + list(class_path.glob('*.png')) # Assuming common image formats
    image_counts[class_name] = len(images)
    total_images += len(images)

print(f"Total images in dataset: {total_images}")
print("Images per class:")
for class_name, count in image_counts.items():
    print(f"  {class_name}: {count}")

# Check for imbalance
min_images = min(image_counts.values())
max_images = max(image_counts.values())
print(f"Min images per class: {min_images}")
print(f"Max images per class: {max_images}")
if max_images / min_images > 1.5: # Simple heuristic for imbalance
    print("Dataset shows signs of imbalance.")
else:
    print("Dataset appears relatively balanced.")

# Sample images from each class and image dimensions
print("\nSample images from each class and example dimensions:")
plt.figure(figsize=(15, 10))
image_dimensions = []

for i, class_name in enumerate(class_names):
    class_path = data_dir / class_name
    images_in_class = list(class_path.glob('*.jpg')) + list(class_path.glob('*.png'))

    if images_in_class:
        # Get a sample image path
        sample_image_path = images_in_class[0]
        img = Image.open(sample_image_path)
        image_dimensions.append(img.size) # (width, height)

        ax = plt.subplot(2, num_classes // 2 + (num_classes % 2), i + 1)
        plt.imshow(img)
        plt.title(f"{class_name}\n{img.size[0]}x{img.size[1]}")
        plt.axis("off")

plt.tight_layout()
plt.show()

# Analyze image dimensions more thoroughly
if image_dimensions:
    widths = [dim[0] for dim in image_dimensions]
    heights = [dim[1] for dim in image_dimensions]
    print(f"Average image width: {np.mean(widths):.2f}")
    print(f"Average image height: {np.mean(heights):.2f}")
    print(f"Min image dimensions: {min(widths)}x{min(heights)}")
    print(f"Max image dimensions: {max(widths)}x{max(heights)}")
else:
    print("No images found to determine dimensions.")


# --- Task 3: Image Preprocessing ---
print("\n--- Task 3: Image Preprocessing ---")

# Define image parameters
IMG_HEIGHT = 128 # You can adjust this based on your dataset and computational resources
IMG_WIDTH = 128
BATCH_SIZE = 32

# Create TensorFlow datasets for training, validation, and testing
# Resizing and normalization (rescale=1./255) are handled here.
# We'll split the dataset into training and validation, and then create a test set.

# Create a full dataset first to then split
full_dataset = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    labels='inferred',
    label_mode='int',
    image_size=(IMG_HEIGHT, IMG_WIDTH),
    interpolation='nearest',
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=42
)

# Determine the number of batches for each split
val_size = int(0.2 * tf.data.experimental.cardinality(full_dataset))
test_size = int(0.1 * tf.data.experimental.cardinality(full_dataset))
train_size = tf.data.experimental.cardinality(full_dataset) - val_size - test_size

# Split the dataset
train_ds = full_dataset.skip(val_size + test_size)
val_ds = full_dataset.skip(test_size).take(val_size)
test_ds = full_dataset.take(test_size)

# Optional: Data augmentation. This can also be added as a layer in the model later.
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal", input_shape=(IMG_HEIGHT, IMG_WIDTH, 3)),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
])

# Apply caching and prefetching for performance
def prepare(ds, shuffle=False, augment=False):
    # Rescale operation is already done by image_dataset_from_directory (pixel values are 0-255)
    # We need to explicitly rescale them to [0,1]
    normalization_layer = tf.keras.layers.Rescaling(1./255)
    ds = ds.map(lambda x, y: (normalization_layer(x), y), num_parallel_calls=tf.data.AUTOTUNE)

    if shuffle: # Shuffle only training data
        ds = ds.shuffle(1000)
    if augment: # Augment only training data
        ds = ds.map(lambda x, y: (data_augmentation(x, training=True), y), num_parallel_calls=tf.data.AUTOTUNE)

    return ds.cache().prefetch(buffer_size=tf.data.AUTOTUNE)

train_ds = prepare(train_ds, shuffle=True, augment=True)
val_ds = prepare(val_ds)
test_ds = prepare(test_ds)

print(f"Training dataset batches: {tf.data.experimental.cardinality(train_ds)}")
print(f"Validation dataset batches: {tf.data.experimental.cardinality(val_ds)}")
print(f"Test dataset batches: {tf.data.experimental.cardinality(test_ds)}")

print("Dataset preprocessing complete. Data is ready for model creation.")

# Make class_names available for subsequent steps (e.g., model output layer)
_CLASS_NAMES = class_names
_NUM_CLASSES = num_classes
_IMG_HEIGHT = IMG_HEIGHT
_IMG_WIDTH = IMG_WIDTH


TensorFlow version: 2.20.0

--- Part 1: Downloading and Extracting Dataset ---
Successfully downloaded dataset.zip


BadZipFile: File is not a zip file